# colab_00_audit — prove the synthetic router is ungrounded (CPU, 30 min, FREE)

**Goal:** single figure + DECISION.json that freezes the query list. No training here.

Free-only. Runs on Colab CPU, no API keys needed.

Anti-spin rule: this notebook FREEZES `query_list.json` (sha256). NB01 must assert the hash.

In [ ]:
# 0. Setup — self-recovering: finds repo, unzips upload, or clones (public or private via token)
import os, sys, json, hashlib, subprocess, zipfile
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE = Path('/content/drive/MyDrive/Research/colab_ckpt')
    DRIVE.mkdir(parents=True, exist_ok=True)
    print('Drive:', DRIVE)
except Exception as e:
    print('No Colab Drive (local run?):', e)
    DRIVE = Path('./colab_ckpt')
    DRIVE.mkdir(parents=True, exist_ok=True)

def _find_backend():
    """Return BACKEND dir (containing src/gateway/service.py) or None."""
    cands = [Path('/content/code-mixed-gateway/backend'), Path('/content/Research/backend'),
             Path('/content/backend'), Path.cwd()/'backend', Path.cwd()/'code-mixed-gateway'/'backend']
    try:
        for p in Path('/content').iterdir():
            if p.is_dir():
                cands += [p, p/'backend']
    except Exception: pass
    for c in cands:
        if (c/'src'/'gateway'/'service.py').exists(): return c
    # deep search one level for upload_tmp style dirs
    try:
        for p in Path('/content').glob('*/**/service.py'):
            if p.parent.parent.name == 'gateway' and p.parent.name == 'gateway':
                be = p.parent.parent.parent  # .../src/gateway/service.py -> backend?
                # walk up to dir containing 'src'
                q = p.parent
                while q != Path('/content') and q.name != 'src': q = q.parent
                be = q.parent
                if (be/'src'/'gateway'/'service.py').exists(): return be
    except Exception: pass
    return None

def _try_unzip_uploads():
    roots = [Path('/content')]
    try: roots.append(DRIVE)
    except Exception: pass
    for root in roots:
        try: zips = list(root.glob('backend*.zip'))
        except Exception: continue
        for z in zips:
            dest = Path('/content/repo_upload')
            if (dest/'backend'/'src'/'gateway'/'service.py').exists(): return True
            print(f'unzipping {z} -> {dest} ...')
            dest.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(z) as zf: zf.extractall(dest)
            # handle zip containing backend/ at top or nested one level
            if not (dest/'backend'/'src'/'gateway'/'service.py').exists():
                for sub in dest.iterdir():
                    if (sub/'backend'/'src'/'gateway'/'service.py').exists():
                        import shutil
                        shutil.move(str(sub/'backend'), str(dest/'backend_tmp'))
                        shutil.move(str(dest/'backend_tmp'), str(dest/'backend'))
                        break
            print('unzip done')
            return True
    return False

def _try_clone():
    dest = Path('/content/code-mixed-gateway')
    if dest.exists(): return dest.exists()
    token = ''
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN') or ''
    except Exception: token = os.environ.get('GITHUB_TOKEN','') or ''
    url = ('https://'+token+'@github.com/tusharsaharan/code-mixed-gateway.git') if token else 'https://github.com/tusharsaharan/code-mixed-gateway.git'
    print('cloning (token=%s) ...' % ('yes' if token else 'no'))
    r = subprocess.run(['git','clone',url,str(dest)], capture_output=True, text=True)
    print('clone rc=', r.returncode)
    if r.returncode != 0: print((r.stderr or '')[-500:])
    return dest.exists()

BACKEND = _find_backend()
if BACKEND is None and _try_unzip_uploads(): BACKEND = _find_backend()
if BACKEND is None and _try_clone(): BACKEND = _find_backend()
print('BACKEND =', BACKEND)
assert BACKEND is not None, ('REPO NOT FOUND. Fastest fix (2 min): on your PC zip the backend folder '
    '(Research/backend -> backend.zip), drag backend.zip into Colab Files panel (folder icon, left), then '
    'Cell > Run this cell again. Alternative: make the GitHub repo public, or add a GITHUB_TOKEN secret.')
REPO = BACKEND.parent
DATA = BACKEND/'data'
print('REPO =', REPO, '| DATA exists:', DATA.exists())

%pip install -q pydantic pydantic-settings numpy httpx 2>&1 | tail -1
src = str(BACKEND/'src')
if src not in sys.path: sys.path.insert(0, src)
import gateway
print('import gateway OK from', gateway.__file__)


In [ ]:
# 1. Load committed data (no API)
import json
from pathlib import Path
# REPO/BACKEND/DATA already set in Cell 0 (robust finder) — reuse
# BACKEND already set in Cell 0
# DATA already set in Cell 0

def load_jsonl(p):
    rows = [json.loads(l) for l in open(p, encoding='utf-8') if l.strip()] if Path(p).exists() else []
    print(f'{p}: {len(rows)} rows')
    return rows

bench_sample = load_jsonl(DATA / 'benchmark.sample.jsonl')
seeds = load_jsonl(DATA / 'seed_hinglish.jsonl')
humans = load_jsonl(DATA / 'seed_human.jsonl')
print('bench cols:', list(bench_sample[0].keys()) if bench_sample else None)

In [ ]:
# 2. Score every query with CURRENT heuristic DifficultyScorer + show synthetic calibration gap
from gateway.modules.m4_router.difficulty import DifficultyScorer
from gateway.service import default_calibration
from gateway.modules.m1_pipeline.hinglish import code_mix_ratio

scorer = DifficultyScorer()
texts = [r.get('original', r.get('text','')) for r in (bench_sample + seeds[:50] + humans)]
scores = [scorer.score(t) for t in texts]
cms = [code_mix_ratio(t) for t in texts]

syn = default_calibration(n=2000, seed=42)
syn_scores = [s.nonconformity for s in syn]
syn_fail = [not s.cheap_success for s in syn]
print(f'real-text heuristic scores: mean={sum(scores)/max(1,len(scores)):.3f} n={len(scores)}')
print(f'synthetic calib scores: mean={sum(syn_scores)/len(syn_scores):.3f} fail_rate={sum(syn_fail)/len(syn_fail):.3f}')
print('GAP: synthetic fail=0.30 by construction (service.py:34 rng.random()<0.30). Real fail rate unknown -> must label in NB01.')

In [ ]:
# 3. Plot + freeze query_list.json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

plt.figure(figsize=(10,4))
plt.subplot(1,2,1); plt.hist(scores, bins=20); plt.title('heuristic d(x) on real texts'); plt.xlabel('difficulty')
plt.subplot(1,2,2); plt.hist(syn_scores, bins=20); plt.title('synthetic calib nonconformity'); plt.xlabel('score')
plt.tight_layout(); plt.savefig(str(DRIVE/'fig00_audit.png'), dpi=120); print('saved fig00_audit.png')

# Freeze: query list = bench_sample ids + seed ids + human ids (dedup). NB01 expands ONLY via approved paraphrase fn.
import hashlib
qlist = sorted({r.get('id', f'row-{i}') for i, r in enumerate(bench_sample + seeds + humans)})
blob = json.dumps(qlist, sort_keys=True).encode()
sha = hashlib.sha256(blob).hexdigest()
(DRIVE/'query_list.json').write_text(json.dumps({'ids': qlist, 'sha256': sha, 'n': len(qlist)}, indent=2))
(DRIVE/'DECISION_00.json').write_text(json.dumps({'stage':'00_audit','query_sha256':sha,'n_queries':len(qlist),'heuristic_mean':round(sum(scores)/max(1,len(scores)),4),'synthetic_fail_rate':0.30,'next':'NB01 must assert query_sha256'}, indent=2))
print('FROZEN query_sha256:', sha, 'n=', len(qlist))